# Task 4 — Retail Data Integration

**Topic:** Combine Excel, SQL, and Python  
**Dataset:** Indian FMCG Retail Sales Customer Inventory (2024)

### Workflow
1. Load and clean the retail dataset using Python.
2. Save the cleaned CSV.
3. Perform SQL analysis in MySQL (documented separately in `Retail_Data_Integration_Task_4.sql`).
4. Create the Excel analysis workbook using Python/Pandas.


## 1. Import Libraries and Load Raw Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Indian FMCG Retail Sales Customer Inventory (2024).csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

## 2. Check Missing Values

In [ ]:
df.isnull().sum()

In [ ]:
missing_value = df.isnull().sum()
missing_value[missing_value > 0]

## 3. Check Duplicate Rows

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

## 4. Inspect Dataset Structure

In [ ]:
df.info()

## 5. Convert Invoice Date to Datetime

In [ ]:
df["Invoice_Date"] = pd.to_datetime(
    df["Invoice_Date"],
    errors="coerce"
)

print(df["Invoice_Date"].dtype)

## 6. Handle Missing Customer Age

In [ ]:
df["Customer_Age"] = df["Customer_Age"].fillna(
    df["Customer_Age"].median()
)

print(
    "Missing Customer_Age:",
    df["Customer_Age"].isnull().sum()
)

## 7. Handle Missing Customer Gender

In [ ]:
df["Customer_Gender"] = df["Customer_Gender"].fillna(
    df["Customer_Gender"].mode()[0]
)

print(
    "Missing Customer_Gender:",
    df["Customer_Gender"].isnull().sum()
)

## 8. Verify Missing Values After Cleaning

In [ ]:
print("Total missing values:", df.isnull().sum().sum())

## 9. Check Invalid Numeric Values

In [ ]:
print("Units <= 0:", (df["Units"] <= 0).sum())
print("Cost Price <= 0:", (df["Cost_Price"] <= 0).sum())
print("Selling Price <= 0:", (df["Selling_Price"] <= 0).sum())
print("Revenue <= 0:", (df["Revenue"] <= 0).sum())

In [ ]:
print("Margin_% below 0:", (df["Margin_%"] < 0).sum())
print("Margin_% above 100:", (df["Margin_%"] > 100).sum())

## 10. Final Cleaning Verification

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## 11. Before vs After Comparison

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before": [
        100000,
        21,
        45129,
        0
    ],
    "After": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})

comparison

## 12. Save Cleaned CSV

In [ ]:
df.to_csv("cleaned_retail_data.csv", index=False)

print("Cleaned CSV created successfully!")

## 13. SQL Analysis

The cleaned dataset was imported into MySQL and analyzed using SQL.

The SQL queries are saved separately in:

`Retail_Data_Integration_Task_4.sql`

The main analyses include:
- Total revenue
- Total cost and margin
- Revenue by category
- Revenue by city
- Revenue by sales channel
- Revenue by brand
- Average order value
- Revenue by payment mode
- Revenue by store format
- Monthly revenue
- Category and channel revenue


## 14. Load Cleaned Data for Excel Analysis

In [ ]:
df = pd.read_csv("cleaned_retail_data.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

## 15. Summary Metrics

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Cost",
        "Total Margin",
        "Average Order Value"
    ],
    "Value": [
        round(df["Revenue"].sum(), 2),
        round(df["Cost"].sum(), 2),
        round(df["Margin"].sum(), 2),
        round(df["Revenue"].sum() / df["Invoice_ID"].nunique(), 2)
    ]
})

summary

## 16. Revenue by Category

In [ ]:
category = (
    df.groupby("Category", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

category

## 17. Revenue by City

In [ ]:
city = (
    df.groupby("City", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

city

## 18. Revenue by Channel

In [ ]:
channel = (
    df.groupby("Channel", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

channel

## 19. Revenue by Brand

In [ ]:
brand = (
    df.groupby("Brand", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

brand

## 20. Monthly Revenue

In [ ]:
df["Invoice_Date"] = pd.to_datetime(
    df["Invoice_Date"],
    errors="coerce"
)

monthly = (
    df.groupby(df["Invoice_Date"].dt.to_period("M"))["Revenue"]
    .sum()
    .reset_index()
)

monthly["Invoice_Date"] = monthly["Invoice_Date"].astype(str)
monthly = monthly.rename(columns={"Invoice_Date": "Month"})
monthly["Revenue"] = monthly["Revenue"].round(2)

monthly

## 21. Revenue by Store Format

In [ ]:
store = (
    df.groupby("Store_Format", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

store

## 22. Revenue by Category and Channel

In [ ]:
category_channel = (
    df.groupby(["Category", "Channel"], as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values(
        ["Category", "Revenue"],
        ascending=[True, False]
    )
)

category_channel

## 23. Create Final Excel Workbook

In [ ]:
# Install openpyxl once if it is not already installed:
# !pip install openpyxl

with pd.ExcelWriter(
    "Retail_Data_Analysis.xlsx",
    engine="openpyxl"
) as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    category.to_excel(
        writer,
        sheet_name="Revenue by Category",
        index=False
    )
    city.to_excel(
        writer,
        sheet_name="Revenue by City",
        index=False
    )
    channel.to_excel(
        writer,
        sheet_name="Revenue by Channel",
        index=False
    )
    brand.to_excel(
        writer,
        sheet_name="Revenue by Brand",
        index=False
    )
    monthly.to_excel(
        writer,
        sheet_name="Monthly Revenue",
        index=False
    )
    store.to_excel(
        writer,
        sheet_name="Store Format",
        index=False
    )
    category_channel.to_excel(
        writer,
        sheet_name="Category & Channel",
        index=False
    )

print("Excel file created successfully!")

## 24. Verify Excel File

In [ ]:
import os

print(
    "Retail_Data_Analysis.xlsx exists:",
    os.path.exists("Retail_Data_Analysis.xlsx")
)